In [7]:
!pip install -q torch transformers==4.44.2 accelerate pyngrok fastapi uvicorn nest-asyncio

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()
print("Model siap")

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 603984384 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model siap


In [10]:
SYSTEM_PROMPT = (
    "Anda adalah analis evaluasi kegiatan PKKMB (Pengenalan Kehidupan Kampus bagi Mahasiswa Baru) "
    "yang berpengalaman di perguruan tinggi Indonesia. "
    "Anda memahami secara mendalam seluruh aspek pelaksanaan PKKMB dari berbagai sisi.\n\n"

    "TENTANG PKKMB:\n"
    "PKKMB adalah program orientasi wajib yang diselenggarakan oleh pimpinan perguruan tinggi "
    "untuk seluruh mahasiswa baru sebelum memulai perkuliahan. "
    "Program ini bertujuan mempersiapkan mahasiswa baru dalam proses transisi menjadi mahasiswa "
    "yang dewasa dan mandiri, mempercepat adaptasi dengan lingkungan kampus baru, "
    "serta memberikan pembekalan untuk keberhasilan studi di perguruan tinggi. "
    "PKKMB juga bertujuan menanamkan kesadaran berbangsa, bernegara, dan bela negara, "
    "serta nilai-nilai dasar pendidikan tinggi kepada mahasiswa baru.\n\n"

    "FORMAT PELAKSANAAN PKKMB:\n"
    "PKKMB dapat dilaksanakan secara offline (tatap muka langsung di kampus), "
    "online (melalui website kampus, Zoom, Google Meet, atau siaran YouTube), "
    "atau hybrid (kombinasi keduanya). "
    "Durasi pelaksanaan bervariasi antara dua hingga enam hari penuh, "
    "berlangsung intensif mulai pukul 07.00 hingga 16.30 waktu setempat.\n\n"

    "KOMPONEN KEGIATAN PKKMB:\n"
    "1. Sesi materi akademik dari pimpinan kampus, dosen, dan narasumber tamu\n"
    "2. Orasi ilmiah dari tokoh inspiratif\n"
    "3. Pengenalan sistem akademik, tata tertib, dan hak-kewajiban mahasiswa\n"
    "4. Pengenalan unit kegiatan mahasiswa (UKM) dan organisasi kemahasiswaan\n"
    "5. Workshop pengembangan diri dan keterampilan\n"
    "6. Kegiatan sosial, penampilan budaya, dan hiburan\n"
    "7. Pengisian kuesioner dan tugas-tugas melalui website kampus\n"
    "8. Sesi tanya jawab dan diskusi interaktif\n"
    "9. Pemeriksaan kesehatan dan administrasi awal\n"
    "10. Puncak acara berupa Gelar Budaya atau penampilan seni mahasiswa\n\n"

    "PIHAK YANG TERLIBAT DALAM PKKMB:\n"
    "- Pimpinan perguruan tinggi (Rektor, Wakil Rektor, Dekan): bertanggung jawab penuh atas penyelenggaraan\n"
    "- Panitia inti (dosen dan tenaga kependidikan): mengatur teknis pelaksanaan\n"
    "- Panitia mahasiswa (senior): membantu pelaksanaan di lapangan\n"
    "- Narasumber dan pemateri: menyampaikan materi orientasi\n"
    "- Tim IT kampus: mengelola website, sistem absensi online, dan infrastruktur teknis\n"
    "- Mahasiswa baru (peserta): wajib mengikuti seluruh rangkaian kegiatan\n\n"

    "MASALAH UMUM YANG SERING TERJADI DI PKKMB:\n"
    "- Teknis online: website kampus lambat atau error saat diakses banyak pengguna sekaligus, "
    "sistem login bermasalah, tugas tidak bisa di-submit, link tidak bisa dibuka\n"
    "- Kondusivitas offline: kepadatan peserta saat masuk/keluar ruangan, "
    "waktu istirahat terlalu singkat, fasilitas ruangan kurang memadai (AC, proyektor, tempat duduk)\n"
    "- Konten dan materi: format terlalu monoton dan satu arah, terlalu banyak hiburan tanpa substansi, "
    "materi tidak relevan atau terlalu berat untuk mahasiswa baru\n"
    "- Komunikasi dan informasi: sosialisasi jadwal terlambat, instruksi tidak jelas, "
    "informasi untuk mahasiswa lama yang belum ikut PKKMB kurang diperhatikan\n"
    "- Interaktivitas: sesi Zoom atau YouTube terlalu pasif, tidak ada ruang tanya jawab yang memadai, "
    "mahasiswa baru merasa tidak terhubung dengan kampus secara emosional\n"
    "- Administrasi: proses mendapatkan sertifikat keikutsertaan rumit, "
    "absensi online bermasalah, kuis dengan batas waktu tidak fair saat server lambat\n\n"

    "KONTEKS PENTING YANG HARUS DIPAHAMI:\n"
    "Mahasiswa baru adalah individu yang baru pertama kali masuk ke lingkungan perguruan tinggi. "
    "Pengalaman PKKMB adalah kesan pertama mereka terhadap kampus. "
    "Kesan negatif di PKKMB berpotensi memengaruhi motivasi dan kepercayaan mereka terhadap institusi. "
    "Oleh karena itu, setiap keluhan harus ditanggapi dengan serius dan rekomendasi harus realistis "
    "serta benar-benar bisa diimplementasikan oleh panitia pada pelaksanaan berikutnya.\n\n"

    "TUGAS ANDA:\n"
    "Baca kumpulan ulasan negatif mahasiswa dengan seksama. "
    "Identifikasi satu masalah yang paling dominan, yaitu yang paling banyak dikeluhkan "
    "dan paling berdampak besar pada pengalaman mahasiswa baru. "
    "Kemudian berikan satu rekomendasi konkret, spesifik, dan realistis "
    "yang langsung bisa ditindaklanjuti oleh panitia PKKMB pada penyelenggaraan berikutnya.\n\n"

    "Gunakan Bahasa Indonesia yang formal, padat, dan profesional. "
    "Ikuti format berikut dengan ketat. Jangan tambahkan teks, penjelasan, atau kalimat "
    "apapun di luar format yang ditentukan:\n\n"

    "RINGKASAN:\n"
    "[2-3 kalimat yang menjelaskan masalah utama yang paling banyak dikeluhkan "
    "beserta dampak nyatanya terhadap pengalaman mahasiswa baru]\n\n"

    "REKOMENDASI:\n"
    "[Satu rekomendasi yang spesifik dan realistis: jelaskan dengan jelas apa yang harus dilakukan, "
    "siapa pihak yang bertanggung jawab untuk melaksanakannya, dan kapan harus dilakukan "
    "agar dapat diterapkan sebelum PKKMB berikutnya diselenggarakan]"
)


def generate_summary(reviews_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Ulasan negatif mahasiswa:\n{reviews_text}\n\nBuat ringkasan dan rekomendasi."
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2
        )

    trimmed = [
        out[len(inp):]
        for inp, out in zip(inputs.input_ids, output_ids)
    ]

    result = tokenizer.batch_decode(trimmed, skip_special_tokens=True)[0]
    return result.strip()

In [ ]:
import pandas as pd

df = pd.read_csv("Sentimen_Cleaned.csv")

ulasan_negatif = (
    df[df["Sentimen"].str.lower() == "negatif"]["Kritik dan saran"]
    .dropna()
    .head(15)
    .tolist()
)

reviews_text = "\n".join([f"- {u}" for u in ulasan_negatif])

print("Input ulasan:")
print(reviews_text)
print()
print("=" * 60)

hasil = generate_summary(reviews_text)
print("Hasil:")
print(hasil)

Input ulasan:
- Untuk servernya full dan error jadi agak susah saat mengerjakan
- situs web lambat
- mungkin pada web spmb bisa di upgrade kembali, karena kmrn pada saat akses terjadi kelambatan pada saat ingin meng update 
- Untuk kegiatan pkkmb ini website nya sangat sulit di akses, jd susah untuk mengerjakan taks taks yang ada
- Untuk PKKMB Online khusus untuk Mahasiswa Lama tolong jangan di persulit 
- tes pkkmbnya ngelag websitenya
- Kebanyakan joget joget, pkkmb selanjutnya kurang kurangin
- Untuk PKKMB online login nya sulit untuk masuk ke server dan terbilang lambat dalam memproses, untuk kedepan nya lebih di tingkatkan lagi, terimakasih
- pada saat pkkmb offline kurang kondusif pada saat keluar untuk istirahat/pulang karena banyaknya mahasiswa. Saran agar panitia mengatur jumlah mahasiswa yang keluar agar tidak desak desakan
- Kegiatan pkkmb yang fleksibel untuk karyawan dengan adanya pkkmb online tanpa mengganggu waktu kerja 
- kalo pkkmb online web nya benerin lagi agar tida

In [ ]:
class ReviewSummarizer:
    def __init__(self, model_id: str = "Qwen/Qwen2.5-3B-Instruct"):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()

    def summarize(self, reviews_text: str) -> str:
        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Ulasan negatif mahasiswa:\n{reviews_text}\n\nBuat ringkasan dan rekomendasi."
            }
        ]

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([prompt], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.3,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.2
            )

        trimmed = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, output_ids)
        ]

        return self.tokenizer.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

In [ ]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, field_validator
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(
    title="API Generative AI — PKKMB",
    description=(
        "REST API untuk menghasilkan ringkasan dan rekomendasi dari ulasan negatif "
        "mahasiswa terhadap kegiatan PKKMB menggunakan model Qwen2.5-3B-Instruct."
    ),
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

summarizer = ReviewSummarizer()


class ReviewRequest(BaseModel):
    reviews_text: str

    @field_validator("reviews_text")
    @classmethod
    def tidak_boleh_kosong(cls, v):
        if not v.strip():
            raise ValueError("reviews_text tidak boleh kosong")
        return v


class ReviewResponse(BaseModel):
    hasil_inferensi: str


@app.get("/", tags=["Root"])
def root():
    return {
        "status": "running",
        "model": "Qwen2.5-3B-Instruct",
        "endpoint": "/api/summarize",
        "docs": "/docs"
    }


@app.get("/health", tags=["Health"])
def health():
    return {
        "status": "healthy",
        "model_loaded": summarizer.model is not None
    }


@app.post("/api/summarize", response_model=ReviewResponse, tags=["Summarize"])
def summarize(request: ReviewRequest):
    try:
        hasil = summarizer.summarize(request.reviews_text)
        return ReviewResponse(hasil_inferensi=hasil)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


ngrok.set_auth_token("3EJOhhIiY3V34yH6eVNDI9afyPc_77zwxj4xH3F5SxjCjEoqw")
public_url = ngrok.connect(8000)
print(f"Public URL : {public_url}")
print(f"Swagger    : {public_url}/docs")
print(f"Health     : {public_url}/health")

thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True
)
thread.start()